# Transformer-based NER

In [1]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate the great-circle distance (in kilometers) between two points on Earth."""
    R = 6371  # Earth radius in kilometers
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda/2)**2
    distance = 2 * R * math.asin(math.sqrt(a))
    return distance

# Node coordinates from GCXO_Nodes_Def.csv
nodes = {
    'Rwy_12_001': (28.487889, -16.357273),
    'Rwy_12_002': (28.485553, -16.350255),
    'Rwy_12_003': (28.483932, -16.345360),
    'Rwy_12_004': (28.482443, -16.340874),
    'Rwy_12_005': (28.481667, -16.338548),
    'Rwy_12_006': (28.477426, -16.325730),
    'Txy_C0_001': (28.488761, -16.356464),
    'Txy_C0_002': (28.488325, -16.357476)
}

# Calculate distances for each link
links = [
    ('Txy_C0_001', 'Txy_C0_002'),
    ('Txy_C0_002', 'Rwy_12_001'),
    ('Rwy_12_001', 'Rwy_12_002'),
    ('Rwy_12_002', 'Rwy_12_003'),
    ('Rwy_12_003', 'Rwy_12_004'),
    ('Rwy_12_004', 'Rwy_12_005'),
    ('Rwy_12_005', 'Rwy_12_006'),
    ('Rwy_12_006', 'Rwy_12_005')
]

print("Actual link distances in kilometers:")
for link in links:
    node1, node2 = link
    lat1, lon1 = nodes[node1]
    lat2, lon2 = nodes[node2]
    distance = haversine_distance(lat1, lon1, lat2, lon2)
    print(f"  {node1} -> {node2}: {distance:.4f} km")

Actual link distances in kilometers:
  Txy_C0_001 -> Txy_C0_002: 0.1101 km
  Txy_C0_002 -> Rwy_12_001: 0.0524 km
  Rwy_12_001 -> Rwy_12_002: 0.7334 km
  Rwy_12_002 -> Rwy_12_003: 0.5112 km
  Rwy_12_003 -> Rwy_12_004: 0.4687 km
  Rwy_12_004 -> Rwy_12_005: 0.2432 km
  Rwy_12_005 -> Rwy_12_006: 1.3386 km
  Rwy_12_006 -> Rwy_12_005: 1.3386 km


In [2]:
# ! pip install torch --index-url https://download.pytorch.org/whl/cu118

In [3]:
# ! pip install -U spacy[transformers]

In [4]:
# ! python -m spacy init config config_tr.cfg --lang en --pipeline transformer,ner --optimize efficiency --force -G

In [5]:
! python -m spacy train config_tr.cfg --output ./transformer --paths.train ./train_data.spacy --paths.dev ./val_data.spacy --gpu-id 0

ℹ Saving to output directory: transformer
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
Traceback (most recent call last):
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/spacy/__main__.py", line 4, in <module>
    setup_cli()
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/spacy/cli/_util.py", line 87, in setup_cli
    command(prog_name=COMMAND)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/typer/core.py", line 743, in main
    return _main(
  File "/home/yp6443/anaconda3/e

In [6]:
import spacy, time
from spacy.training import Corpus
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

nlp_ner = spacy.load("./transformer/model-best")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/o Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    # Use ['ents_p']
print(f"Recall:    {scores['ents_r']:.3f}")    # Use ['ents_r']
print(f"F1-Score:  {scores['ents_f']:.3f}")    # Use ['ents_f']

['transformer', 'ner']
=== Tok2Vec w/o Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.833
Recall:    0.658
F1-Score:  0.735


In [7]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.01 seconds
Memory used: 330.21 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE


In [8]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate the great-circle distance (in kilometers) between two points on Earth."""
    R = 6371  # Earth radius in kilometers
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda/2)**2
    distance = 2 * R * math.asin(math.sqrt(a))
    return distance

# Node coordinates from GCXO_Nodes_Def.csv
nodes = {
    'Rwy_12_001': (28.487889, -16.357273),
    'Rwy_12_002': (28.485553, -16.350255),
    'Rwy_12_003': (28.483932, -16.345360),
    'Rwy_12_004': (28.482443, -16.340874),
    'Rwy_12_005': (28.481667, -16.338548),
    'Rwy_12_006': (28.477426, -16.325730),
    'Txy_C0_001': (28.488761, -16.356464),
    'Txy_C0_002': (28.488325, -16.357476)
}

# Calculate distances for each link
links = [
    ('Txy_C0_001', 'Txy_C0_002'),
    ('Txy_C0_002', 'Rwy_12_001'),
    ('Rwy_12_001', 'Rwy_12_002'),
    ('Rwy_12_002', 'Rwy_12_003'),
    ('Rwy_12_003', 'Rwy_12_004'),
    ('Rwy_12_004', 'Rwy_12_005'),
    ('Rwy_12_005', 'Rwy_12_006'),
    ('Rwy_12_006', 'Rwy_12_005')
]

print("Actual link distances in kilometers:")
for link in links:
    node1, node2 = link
    lat1, lon1 = nodes[node1]
    lat2, lon2 = nodes[node2]
    distance = haversine_distance(lat1, lon1, lat2, lon2)
    print(f"  {node1} -> {node2}: {distance:.4f} km")

Actual link distances in kilometers:
  Txy_C0_001 -> Txy_C0_002: 0.1101 km
  Txy_C0_002 -> Rwy_12_001: 0.0524 km
  Rwy_12_001 -> Rwy_12_002: 0.7334 km
  Rwy_12_002 -> Rwy_12_003: 0.5112 km
  Rwy_12_003 -> Rwy_12_004: 0.4687 km
  Rwy_12_004 -> Rwy_12_005: 0.2432 km
  Rwy_12_005 -> Rwy_12_006: 1.3386 km
  Rwy_12_006 -> Rwy_12_005: 1.3386 km


In [9]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import glob
test_file = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
file_idx = 1

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [10]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

with open(test_file[file_idx], 'r') as file:
    for line in file:
        doc = nlp_ner(line.strip())
        spacy.displacy.render(doc, style="ent", jupyter=True)

In [11]:
nlp_ner = spacy.load("./transformer/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec w/ Heuristics ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

['transformer', 'ner', 'entity_ruler']
=== Tok2Vec w/ Heuristics ===
=== NER Evaluation Metrics ===
Precision: 0.791
Recall:    0.796
F1-Score:  0.793


In [12]:
from memory_profiler import memory_usage

def test_inference(text: str):
    
    nlp_ner = spacy.load("./transformer/model-best")
    ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
    )
    ruler.from_disk("entity_rulers.jsonl")

    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")

    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.02 seconds
Memory used: 302.91 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding point C1 : DESTINATION


# Train with entity rule spans

In [13]:
! python -m spacy train config_tr.cfg \
    --output ./transformer_rules \
        --paths.train ./augmented_train_data.spacy \
            --paths.dev ./val_data.spacy \
                --gpu-id 0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ℹ Saving to output directory: transformer_rules
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
Traceback (most recent call last):
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/spacy/__main__.py", line 4, in <module>
    setup_cli()
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/spacy/cli/_util.py", line 87, in setup_cli
    command(prog_name=COMMAND)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
  File "/home/yp6443/anaconda3/envs/nlp/lib/python3.10/site-packages/typer/core.py", line 743, in main
    return _main(
  File "/home/yp6443/anaco

In [14]:
from spacy.training import Corpus

nlp_ner = spacy.load("./transformer_rules/model-best")

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")    

=== Tok2Vec trained w/ Heuristics w/o Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.850
Recall:    0.743
F1-Score:  0.793


In [15]:
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.01 seconds
Memory used: 414.59 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE


In [16]:
nlp_ner = spacy.load("./transformer_rules/model-best")

ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
)
ruler.from_disk("entity_rulers.jsonl")
print(nlp_ner.pipe_names)

# Load test data (adjust path to your test data)
test_data_path = "/home/yp6443/research/nlp/taxigen/test_data.spacy"

# If your test data is already in .spacy format:
corpus = Corpus(test_data_path)
test_examples = corpus(nlp_ner)

# Evaluate the model
scores = nlp_ner.evaluate(test_examples)

# Print NER metrics
print("=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===")
print("=== NER Evaluation Metrics ===")
print(f"Precision: {scores['ents_p']:.3f}")    
print(f"Recall:    {scores['ents_r']:.3f}")
print(f"F1-Score:  {scores['ents_f']:.3f}")

['transformer', 'ner', 'entity_ruler']
=== Tok2Vec trained w/ Heuristics w/ Heuristics override ===
=== NER Evaluation Metrics ===
Precision: 0.842
Recall:    0.875
F1-Score:  0.858


In [17]:
import spacy
from memory_profiler import memory_usage
import time

def test_inference(text: str):
    nlp_ner = spacy.load("./transformer/model-best")
    ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": False}  
    )
    ruler.from_disk("entity_rulers.jsonl")

    start_time = time.time()
    doc = nlp_ner(text)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")
    # Return something if needed
    return doc

# 1) Measure memory before/after running the function
test_text = "Japan Air 179, Tokyo Tower, good evening, number 3, taxi to holding point C1."

start_mem = memory_usage()[0]  # baseline memory
doc = test_inference(test_text)
end_mem = memory_usage()[0]    # memory after running

print(f"Memory used: {end_mem - start_mem:.2f} MiB\n")

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Time taken: 0.02 seconds
Memory used: 412.16 MiB

Entities found:
 - Japan Air 179 : CALLSIGN
 - taxi : ACSTATE
 - holding point C1 : DESTINATION
